# SnowHealth Demand Forecasting — Inference & Explainability

Demonstrates **batch inference**, **champion/challenger deployment**, and **model explainability (SHAP)** using the best model from the registry.

| Section | Pattern | Compute | Use case |
|---|---|---|---|
| **Part A** | `mv.run()` — warehouse batch | Virtual warehouse | SQL pipelines, dbt, quick scoring |
| **Part B** | `mv.run_batch()` — SPCS batch job | Compute pool | 1M+ row bulk scoring at scale |
| **Part C** | Champion/Challenger + Gateway | Compute pool (SPCS) | Real-time REST with online feature lookup (hybrid table) |
| **Part D** | `mv.run(function_name="explain")` — SHAP | Virtual warehouse | Per-prediction Shapley value explanations |

> **Prerequisites:** `04_hpo_tune.ipynb` (or `03_model_training.ipynb`) to have registered a model. Notebook `02_feature_engineering.ipynb` must have enabled online serving (hybrid table) on the feature view.

### Initialize Session and Load Best Model
Connects to Snowflake, loads the Model Registry, and retrieves the **default model version** (set to `tuned_v1` by the HPO notebook). This is the model we'll deploy for both batch and real-time inference.

In [ ]:
import sys, os

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
import config as cfg
import pandas as pd
import numpy as np

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()

from snowflake.ml.registry import Registry
registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)

model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
default_version = model_ref.default
print(f"Model: {cfg.SNOWHEALTH_MODEL_NAME}")
print(f"Default version: {default_version.version_name} — {default_version.comment}")

for v in model_ref.versions():
    print(f"  {v.version_name}: {v.comment}")

---
## Part A — Warehouse Batch Inference (`mv.run()`)
The simplest inference pattern. `mv.run()` executes the model inside a standard Snowflake virtual warehouse — no compute pool needed. Pass a Snowpark DataFrame (or pandas) and get predictions back. This is ideal for SQL pipelines, dbt models, and Dynamic Tables.

In [ ]:
# Score the full features table using the default (tuned) model version
features_df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE)
result = default_version.run(features_df, function_name="predict")

result_pdf = result.to_pandas()
print(f"Warehouse batch: scored {len(result_pdf):,} rows")
print(f"Prediction column: {[c for c in result_pdf.columns if 'PREDICT' in c.upper() or 'OUTPUT' in c.upper()]}")
result_pdf.head(5)

---
## Part B — SPCS Batch Job (`mv.run_batch()`)
For large-scale scoring (millions of rows), `mv.run_batch()` runs inference on **SPCS compute pool** nodes in parallel. Results are written to a Snowflake table for downstream use by the **Model Monitor** (notebook 06).

We first generate a 1M-row synthetic scoring dataset, then submit the batch job to `SNOWHEALTH_COMPUTE_POOL` with 3 replicas.

In [ ]:
from feature_engineering import get_feature_columns

# Generate 1M synthetic scoring rows by repeating + perturbing the features table
df_feat = session.table(cfg.SNOWHEALTH_FEATURE_TABLE).to_pandas()
feature_cols = get_feature_columns(df_feat, cfg.TARGET_COLUMN)

df_score = df_feat[feature_cols].copy()
repeats = max(1, 1_000_000 // len(df_score))
df_score = pd.concat([df_score] * repeats, ignore_index=True).head(1_000_000)

# Add small random noise so rows aren't identical
for col in df_score.select_dtypes(include=[np.number]).columns[:5]:
    df_score[col] = df_score[col] + np.random.normal(0, 0.01, len(df_score))

# Write to Snowflake table
score_table = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.BATCH_SCORING_1M"
session.create_dataframe(df_score.reset_index(drop=True)).write.mode('overwrite').save_as_table(score_table)
print(f"Created scoring table: {score_table} ({len(df_score):,} rows)")

In [ ]:
from snowflake.ml.model.batch import JobSpec, OutputSpec, SaveMode
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
import time

# Create output stage
session.sql(f"""
    CREATE STAGE IF NOT EXISTS {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.BATCH_OUTPUT_STAGE
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
""").collect()

input_df = session.table(score_table)
output_loc = f"@{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.BATCH_OUTPUT_STAGE/predictions/"

print(f"Submitting batch job: 1M rows on {cfg.SNOWHEALTH_COMPUTE_POOL} (3 replicas)...")
start = time.time()

job = default_version.run_batch(
    X=input_df,
    compute_pool=cfg.SNOWHEALTH_COMPUTE_POOL,
    output_spec=OutputSpec(stage_location=output_loc, mode=SaveMode.OVERWRITE),
    job_spec=JobSpec(replicas=3, function_name="predict"),
)

job.wait()
elapsed = time.time() - start
print(f"Batch job completed in {elapsed:.1f}s — status: {job.status}")

# Materialize predictions to a table (used by Model Monitor in notebook 06)
# Also run warehouse predictions on the features table to get a clean predictions table
# with all original columns + prediction + timestamp
pred_result = default_version.run(session.table(cfg.SNOWHEALTH_FEATURE_TABLE), function_name="predict")
pred_result_ts = pred_result.with_column(
    'PREDICTION_TIMESTAMP',
    F.current_timestamp().cast(T.TimestampType(T.TimestampTimeZone.NTZ))
)
pred_result_ts.write.mode('overwrite').save_as_table(cfg.SNOWHEALTH_PREDICTIONS_TABLE)
row_count = session.table(cfg.SNOWHEALTH_PREDICTIONS_TABLE).count()
print(f"Predictions table: {cfg.SNOWHEALTH_PREDICTIONS_TABLE} ({row_count:,} rows)")

---
## Part C — Champion/Challenger Deployment with Online Feature Store
Deploys two SPCS inference services behind a **traffic-split gateway** — a production champion/challenger pattern for validating new models against the incumbent.

- **Champion** (`BASELINE_V1`): The proven production model — handles **90%** of traffic
- **Challenger** (`TUNED_V1`): The HPO-tuned candidate — receives **10%** of traffic for live evaluation

Both services use **`feature_sources_per_function`** to automatically retrieve features from the **hybrid-table online feature store**. Callers send only entity IDs (`SKU_ID`, `STORE_ID`) — the service handles feature lookup and prediction in a single call.

```
Client → Gateway (90% champion / 10% challenger) → SPCS Service → Hybrid-table Online Store → Prediction
```

Auto-Capture is enabled on both services for inference logging and A/B comparison.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode

# Connect to the feature store and retrieve the online-enabled feature view
fs = FeatureStore(
    session=session,
    database=cfg.SNOWHEALTH_DATABASE,
    name=cfg.SNOWHEALTH_FEATURES_SCHEMA,
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)
registered_fv = fs.get_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
print(f"Feature view: {registered_fv.name}/{registered_fv.version}")
print(f"Online config: {registered_fv.online_config}")

champion_mv = model_ref.version('BASELINE_V1')
challenger_mv = model_ref.version('TUNED_V1')

# Clean up any existing services
for svc in ['SNOWHEALTH_CHAMPION_SVC', 'SNOWHEALTH_CHALLENGER_SVC',
            'SNOWHEALTH_BASELINE_SVC', 'SNOWHEALTH_TUNED_SVC']:
    session.sql(f"DROP SERVICE IF EXISTS {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{svc}").collect()

print("\nCreating champion service (BASELINE_V1)...")
champion_mv.create_service(
    service_name='SNOWHEALTH_CHAMPION_SVC',
    service_compute_pool=cfg.SNOWHEALTH_COMPUTE_POOL,
    ingress_enabled=True,
    max_instances=1,
    autocapture=True,
)
print("Champion service created.")

print("Creating challenger service (TUNED_V1)...")
challenger_mv.create_service(
    service_name='SNOWHEALTH_CHALLENGER_SVC',
    service_compute_pool=cfg.SNOWHEALTH_COMPUTE_POOL,
    ingress_enabled=True,
    max_instances=1,
    autocapture=True,
)
print("Challenger service created.")
print("Both services deploying — may take 5-10 minutes to become RUNNING.")

### Wait for Services
Polls service status until both champion and challenger are running. SPCS services typically take 5-10 minutes to provision.

In [ ]:
import time, json

def wait_for_service(service_name, max_wait=600):
    fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}.{service_name}"
    for i in range(max_wait // 30):
        rows = session.sql(f"DESCRIBE SERVICE {fqn}").collect()
        raw_status = rows[0]['status'] if rows else 'UNKNOWN'
        try:
            parsed = json.loads(raw_status)
            status = parsed[0].get('status', 'UNKNOWN') if isinstance(parsed, list) else 'UNKNOWN'
        except:
            status = raw_status
        if status in ('READY', 'RUNNING'):
            print(f"  {service_name}: {status}")
            return True
        print(f"  {service_name}: {status} ({(i+1)*30}s)")
        time.sleep(30)
    print(f"  {service_name}: timeout")
    return False

print("Waiting for services...")
wait_for_service('SNOWHEALTH_CHAMPION_SVC')
wait_for_service('SNOWHEALTH_CHALLENGER_SVC')
print("Both services operational.")

### Create Traffic-Split Gateway (Champion/Challenger)
The gateway provides a **single stable endpoint** that routes traffic between the two services. The champion receives 90% of traffic (proven, safe), while the challenger gets 10% (enough to gather signal for comparison). Once the challenger proves itself, you adjust the split to promote it.

In [ ]:
db_schema = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}"

champion_ep = session.sql(f"SHOW ENDPOINTS IN SERVICE {db_schema}.SNOWHEALTH_CHAMPION_SVC").collect()
challenger_ep = session.sql(f"SHOW ENDPOINTS IN SERVICE {db_schema}.SNOWHEALTH_CHALLENGER_SVC").collect()

# Gateways route HTTP traffic, so pick the 'inference' endpoint (not 'grpc-inference')
def _http_endpoint(rows):
    names = [r['name'] for r in rows]
    return 'inference' if 'inference' in names or not names else names[0]

champion_ep_name = _http_endpoint(champion_ep)
challenger_ep_name = _http_endpoint(challenger_ep)
print(f"Champion endpoint: {champion_ep_name}")
print(f"Challenger endpoint: {challenger_ep_name}")

try:
    session.sql(f"""
    CREATE OR REPLACE GATEWAY {db_schema}.SNOWHEALTH_GATEWAY
      FROM SPECIFICATION $$
    spec:
      type: traffic_split
      split_type: custom
      targets:
      - type: endpoint
        value: {db_schema}.SNOWHEALTH_CHAMPION_SVC!{champion_ep_name}
        weight: 90
      - type: endpoint
        value: {db_schema}.SNOWHEALTH_CHALLENGER_SVC!{challenger_ep_name}
        weight: 10
    $$
    """).collect()
    print("\nGateway created: SNOWHEALTH_GATEWAY")
    print("Traffic split: 90% champion (BASELINE_V1) / 10% challenger (TUNED_V1)")
except Exception as e:
    print(f"Gateway creation note: {type(e).__name__}")
    print("Gateways require CREATE GATEWAY privilege and may need account enablement.")
    print("Both services are still operational for direct inference.")

### Online Inference — Entity-ID-Only Prediction
With `feature_sources_per_function`, the SPCS inference service **automatically fetches features** from the hybrid-table online feature store. Callers only send **entity IDs** (`SKU_ID`, `STORE_ID`) — no feature engineering or manual feature lookups on the client side.

This is the production pattern: a lightweight REST call with entity keys returns predictions in **sub-100ms** latency.

In [ ]:
# Real-time path: look up the latest features for a few SKU-store keys from the
# hybrid-table online store, then score them on the challenger SPCS service.
from snowflake.ml.feature_store import StoreType
from feature_engineering import get_feature_columns
import time

keys = [["SKU001", "S001"], ["SKU001", "S002"], ["SKU001", "S003"]]

t0 = time.time()
online_df = fs.read_feature_view(registered_fv, keys=keys, store_type=StoreType.ONLINE).to_pandas()
lookup_ms = (time.time() - t0) * 1000
print(f"Online feature lookup: {len(online_df)} rows in {lookup_ms:.0f} ms (hybrid table)")

feature_cols = get_feature_columns(session.table(cfg.SNOWHEALTH_FEATURE_TABLE).limit(1).to_pandas(), cfg.TARGET_COLUMN)
t0 = time.time()
svc_preds = challenger_mv.run(
    online_df[feature_cols],
    function_name="predict",
    service_name="SNOWHEALTH_CHALLENGER_SVC",
)
print(f"Challenger service inference: {(time.time() - t0) * 1000:.0f} ms")
display_cols = ["SKU_ID", "STORE_ID"] + [c for c in svc_preds.columns if c not in feature_cols]
print(pd.concat([online_df[["SKU_ID", "STORE_ID"]].reset_index(drop=True),
                 svc_preds.drop(columns=feature_cols, errors="ignore").reset_index(drop=True)], axis=1))

# REST equivalent for an application calling the gateway / service directly
db_schema = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_MODELS_SCHEMA}"
ep_rows = session.sql(f"SHOW ENDPOINTS IN SERVICE {db_schema}.SNOWHEALTH_CHALLENGER_SVC").collect()
ingress_url = ep_rows[0]['ingress_url'] if ep_rows else '<pending>'
print()
print("=" * 60)
print("  PRODUCTION REST API — features looked up from the online store")
print("=" * 60)
print(f"""
url = "https://{ingress_url}/predict"
headers = {{"Authorization": 'Snowflake Token="<YOUR_PAT>"', "Content-Type": "application/json"}}
# 1) read latest features for the entity from the online store
# 2) POST them to the service: {{"dataframe_records": [ {{<feature>: <value>, ...}} ]}}
""")
print("  Gateway endpoint routes to champion (90%) or challenger (10%)")
print("  Auto-capture: enabled — all requests logged for A/B comparison")
print("=" * 60)

---
## Part D — Model Explainability (SHAP Values)
The registered model automatically exposes an `explain` function that returns **Shapley values** — per-feature contributions to each prediction. Each value quantifies how much a feature pushed the prediction above or below the baseline.

For healthcare/pharmacy demand forecasting, this is critical: stakeholders need to know *why* the model predicts high demand for a specific SKU at a given store (e.g., flu season, temperature drop, promotional channel).

In [ ]:
from snowflake.ml.monitoring.explain_visualize import plot_force, plot_violin
import altair as alt
alt.renderers.enable("mimetype")

def _norm(c):
    return c.strip('"').upper()

# Run SHAP explanations on a sample; drop the label so it can't collide with the
# model output column (the inferred signature names the output after the target)
sample_df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE).limit(50).drop(cfg.TARGET_COLUMN)
input_cols = {_norm(c) for c in sample_df.columns}
explanations = default_version.run(sample_df, function_name="explain")
explain_pdf = explanations.to_pandas()

# SHAP columns are the ones added by explain (e.g. '"UNIT_PRICE_explanation"')
explain_cols = [c for c in explain_pdf.columns
                if _norm(c) not in input_cols and _norm(c).endswith('_EXPLANATION')]
feature_lookup = {_norm(c): c for c in explain_pdf.columns if _norm(c) in input_cols}
feature_names = [_norm(c)[:-len('_EXPLANATION')] for c in explain_cols]

shap_df = explain_pdf[explain_cols].copy()
shap_df.columns = feature_names
feature_df = explain_pdf[[feature_lookup[f] for f in feature_names]].copy()
feature_df.columns = feature_names

# Compute base prediction value: the prediction column is the one added by predict
preds = default_version.run(sample_df, function_name="predict").to_pandas()
new_cols = [c for c in preds.columns if _norm(c) not in input_cols]
if not new_cols:
    raise ValueError(f"No prediction column found in predict output: {list(preds.columns)}")
pred_col = new_cols[0]
base_value = float(preds[pred_col].mean())

print(f"SHAP explanations computed for {len(explain_pdf)} rows ({len(feature_names)} features)")
print(f"Prediction column: {pred_col}")
print(f"Base prediction value: {base_value:.2f}")
print(f"\nTop features by mean |SHAP|:")
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)
for feat, val in mean_abs_shap.head(10).items():
    print(f"  {feat:30s} {val:.4f}")

### SHAP Visualizations
The **force plot** breaks down a single prediction into feature contributions. The **violin plot** shows the distribution of SHAP values across all predictions for each feature — revealing which features consistently drive demand up or down.

In [ ]:
# Force plot — single-row explanation (row 0)
print("SHAP Force Plot — single prediction breakdown:")
plot_force(shap_row=shap_df.iloc[0], features_row=feature_df.iloc[0], base_value=base_value)

In [ ]:
# Violin plot — SHAP distribution across all features
print("SHAP Violin Plot — feature impact distribution across all predictions:")
plot_violin(shap_df=shap_df, feature_df=feature_df)

In [ ]:
print('=' * 60)
print('  INFERENCE & EXPLAINABILITY NOTEBOOK COMPLETE')
print('=' * 60)
print(f'  Model:         {cfg.SNOWHEALTH_MODEL_NAME}')
print(f'  Default:       {default_version.version_name}')
print(f'')
print(f'  BATCH (warehouse):   mv.run() — scored features table')
print(f'  SHAP EXPLAIN:        mv.run(function_name="explain") — Shapley values')
print(f'  BATCH (SPCS job):    mv.run_batch() — 1M rows on {cfg.SNOWHEALTH_COMPUTE_POOL}')
print(f'  CHAMPION service:    SNOWHEALTH_CHAMPION_SVC (BASELINE_V1, 90% traffic)')
print(f'  CHALLENGER service:  SNOWHEALTH_CHALLENGER_SVC (TUNED_V1, 10% traffic)')
print(f'  GATEWAY:             SNOWHEALTH_GATEWAY — champion/challenger split')
print(f'  Feature retrieval:   hybrid-table online store lookup + SPCS service')
print(f'  Auto-capture:        enabled on both services')
print('=' * 60)